In [1]:
from sklearn.datasets import fetch_openml

X_mnist, y_mnist = fetch_openml('mnist_784', return_X_y=True, as_frame=False,
                                parser='auto')

In [2]:
X_mnist.shape

(70000, 784)

In [3]:
y_mnist.shape

(70000,)

In [ ]:
# slicing here excludes the stop index and includes start index
X_train = X_mnist[:50000]
y_train = y_mnist[:50000]

X_train.shape

(50000, 784)

In [8]:
X_valid = X_mnist[50000:60000]
y_valid = y_mnist[50000:60000]
X_valid.shape

(10000, 784)

In [7]:
X_test = X_mnist[60000:]
y_text = y_mnist[60000:]
X_test.shape

(10000, 784)

In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier

from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# Needs Scaling to perform well
svc_clf = make_pipeline(StandardScaler(), SVC(C=1.0, kernel="linear", random_state=42))
rnd_clf = RandomForestClassifier(n_estimators=100, random_state=42)
ext_clf = ExtraTreesClassifier(n_estimators=100)



In [10]:
# Training every single estimator
estimators = [svc_clf, rnd_clf, ext_clf]

for i, est_rec in enumerate(estimators):
    est_rec.fit(X_train, y_train)


In [11]:
single_estimator_scores = []

for i, est_rec in enumerate(estimators):
    validation_score = est_rec.score(X_valid, y_valid)
    single_estimator_scores.append(validation_score)

print(single_estimator_scores)


[0.9276, 0.9736, 0.9738]


In [15]:
for i, est_rec in enumerate(estimators):
    print(est_rec)
    print(single_estimator_scores[i])

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('svc', SVC(kernel='linear', random_state=42))])
0.9276
RandomForestClassifier(random_state=42)
0.9736
ExtraTreesClassifier()
0.9738


In [39]:
# Combining to Voting Classifier
from sklearn.ensemble import VotingClassifier

# Hard Voting
voting_clf = VotingClassifier(
    estimators=[
        ('svc', make_pipeline(StandardScaler(), SVC(C=1.0, kernel="linear", random_state=42)))
        , ('rnd', RandomForestClassifier(n_estimators=100, random_state=42))
        , ('ext', ExtraTreesClassifier(n_estimators=100))
    ]
)



In [40]:
# Training the voting Classifier
voting_clf.fit(X_train, y_train)

,estimators,"[('svc', ...), ('rnd', ...), ...]"
,voting,'hard'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,C,1.0
,kernel,'linear'


In [21]:
# Validating
print(f"Voting Classifier Score: {voting_clf.score(X_valid, y_valid)}")

from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y_valid_encoded = encoder.fit_transform(y_valid)

print("Single Classifier Scores: ")
for name, clf in voting_clf.named_estimators_.items():
    print(name, "=", clf.score(X_valid, y_valid_encoded))

Voting Classifier Score: 0.9744
Single Classifier Scores: 
svc = 0.9276
rnd = 0.9736
ext = 0.9737


In [34]:
# Validating soft voting
# we need to set probability to true to use soft voting (predict_proba)
# Hard Voting
voting_clf_soft = VotingClassifier(
    estimators=[
        ('svc', make_pipeline(StandardScaler(), SVC(C=1.0, kernel="linear", random_state=42, probability=True)))
        , ('rnd', RandomForestClassifier(n_estimators=100, random_state=42))
        , ('ext', ExtraTreesClassifier(n_estimators=100))
    ]
)


In [35]:
voting_clf_soft.voting = "soft"

In [36]:
voting_clf_soft.fit(X_train, y_train)

,estimators,"[('svc', ...), ('rnd', ...), ...]"
,voting,'soft'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,C,1.0
,kernel,'linear'


In [37]:
voting_clf_soft.score(X_valid, y_valid)

0.9691

In [ ]:
# --> Hard Voting wins in this case

print(f"Hard voting Test Score: {voting_clf.score(X_test, y_text)}")



Hard voting Test Score: 0.9698


In [44]:
# Single Estimators:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y_test_encoded = encoder.fit_transform(y_text)

print("Single Classifier Scores: ")
for name, clf in voting_clf.named_estimators_.items():
    print(name, "=", clf.score(X_test, y_test_encoded))

Single Classifier Scores: 
svc = 0.926
rnd = 0.968
ext = 0.9711


In [52]:
# SVC has bad scoring --> maybe performance drops due to this

from sklearn.neighbors import KNeighborsClassifier

voting_clf_non_svc = VotingClassifier(
    estimators=[
        ('ada', KNeighborsClassifier(n_neighbors=5))
        , ('rnd', RandomForestClassifier(n_estimators=100, random_state=42))
        , ('ext', ExtraTreesClassifier(n_estimators=100))
    ]
)

In [53]:
voting_clf_non_svc.fit(X_train, y_train)

,estimators,"[('ada', ...), ('rnd', ...), ...]"
,voting,'hard'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2


In [56]:
voting_clf_non_svc.score(X_test, y_text)

y_test_encoded = encoder.fit_transform(y_text)
print("Single Classifier Scores: ")
for name, clf in voting_clf_non_svc.named_estimators_.items():
    print(name, "=", clf.score(X_test, y_test_encoded))

Single Classifier Scores: 
ada = 0.9664
rnd = 0.968
ext = 0.971


In [57]:
voting_clf_non_svc.score(X_test, y_text)

0.9731

In [66]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler

voting_clf_non_linreg = VotingClassifier(
    estimators=[
        ('logrec', make_pipeline(MinMaxScaler(), LogisticRegression(random_state=42, max_iter=1000, n_jobs=4)))
        , ('rnd', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4))
        , ('ext', ExtraTreesClassifier(n_estimators=100, n_jobs=4))
    ]
)

In [67]:
voting_clf_non_linreg.fit(X_train, y_train)

,estimators,"[('logrec', ...), ('rnd', ...), ...]"
,voting,'hard'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,feature_range,"(0, ...)"
,copy,True
,clip,False
,penalty,'l2'
,dual,False


In [68]:
clf_score = voting_clf_non_linreg.score(X_test, y_text)
print("Score of lin reg voter", ": ", clf_score)

print("Single Estimators: ")
for k, est_name in voting_clf_non_linreg.named_estimators_.items():
    print(est_name, ": ", est_name.score(X_test, y_test_encoded))

Score of lin reg voter :  0.9689
Single Estimators: 
Pipeline(steps=[('minmaxscaler', MinMaxScaler()),
                ('logisticregression',
                 LogisticRegression(max_iter=1000, n_jobs=4, random_state=42))]) :  0.9247
RandomForestClassifier(n_jobs=4, random_state=42) :  0.968
ExtraTreesClassifier(n_jobs=4) :  0.9725
